In [89]:
import pandas as pd
import openpyxl
from pathlib import Path

In [90]:
# Define the folder where your Excel files are stored
folder = Path("./raw_data/")

# Get all .xlsx files in the folder
excel_files = list(folder.glob("*.xlsx"))

# Print to confirm
print(f"Found {len(excel_files)} Excel files:")
for file in excel_files:
    print(file)

Found 4 Excel files:
raw_data/s_raw.xlsx
raw_data/l_raw.xlsx
raw_data/m_raw.xlsx
raw_data/xl_raw.xlsx


In [91]:
all_dataframes = []
for file in excel_files:
    excel = pd.ExcelFile(file)

    for sheet in excel.sheet_names:
        df = pd.read_excel(excel, sheet_name=sheet)
        df["household_id"] = f"{sheet}"
        all_dataframes.append(df)

In [92]:
df_unsorted = pd.concat(all_dataframes, ignore_index=True)
df_unsorted['Datetime'] = pd.to_datetime(df_unsorted['Datetime'],format='%d.%m.%Y %H:%M')

# Sanity check
print(f"Total rows: {len(df_unsorted)}")
print(f"Total columns: {df_unsorted.columns.tolist()}")
print(df_unsorted.head())
print(df_unsorted.tail())

Total rows: 1293745
Total columns: ['Datetime', 'NPS (€/MWh)', 'Total OUT (kWh)', 'Total IN (kWh)', 'household_id']
             Datetime  NPS (€/MWh)  Total OUT (kWh)  Total IN (kWh)  \
0 2025-03-01 00:00:00       164.80            0.920             0.0   
1 2025-03-01 01:00:00       170.08            1.077             0.0   
2 2025-03-01 02:00:00       120.48            1.086             0.0   
3 2025-03-01 03:00:00       117.15            0.797             0.0   
4 2025-03-01 04:00:00       112.42            0.599             0.0   

  household_id  
0          s-1  
1          s-1  
2          s-1  
3          s-1  
4          s-1  
                   Datetime  NPS (€/MWh)  Total OUT (kWh)  Total IN (kWh)  \
1293740 2026-02-28 22:45:00        59.37            5.487             0.0   
1293741 2026-02-28 23:00:00        75.84            0.798             0.0   
1293742 2026-02-28 23:15:00        69.85            5.474             0.0   
1293743 2026-02-28 23:30:00        67.32       

In [93]:
# How many unique households?
#print(f"Unique households: {df_unsorted['household_id'].nunique()}")

# How many rows per household?
#print(df_unsorted.groupby('household_id').size())

# Any missing values?
print(f"Missing values per column:")
print(df_unsorted.isnull().sum())

# Data types — are timestamps and consumption in the right format?
print(f"\nData types:")
print(df_unsorted.dtypes)

Missing values per column:
Datetime           0
NPS (€/MWh)        0
Total OUT (kWh)    0
Total IN (kWh)     0
household_id       0
dtype: int64

Data types:
Datetime           datetime64[us]
NPS (€/MWh)               float64
Total OUT (kWh)           float64
Total IN (kWh)            float64
household_id                  str
dtype: object


In [94]:
mask = (df_unsorted['Datetime'].dt.month == 3) & (df_unsorted['Datetime'].dt.year == 2025)
df_unsorted = df_unsorted[~mask]

In [95]:
df_unsorted.head()

,Datetime,NPS (€/MWh),Total OUT (kWh),Total IN (kWh),household_id
743,2025-04-01 00:00:00,102.39,0.026,0.0,s-1
744,2025-04-01 00:15:00,102.39,0.063,0.0,s-1
745,2025-04-01 00:30:00,102.39,0.021,0.0,s-1
746,2025-04-01 00:45:00,102.39,0.028,0.0,s-1
747,2025-04-01 01:00:00,250.03,0.013,0.0,s-1


In [96]:
df_unsorted = df_unsorted.drop(columns=['Total IN (kWh)', 'NPS (€/MWh)'])

In [97]:
df_unsorted = df_unsorted.rename(columns={
    'Datetime': 'timestamp',
    'Total OUT (kWh)': 'consumption_kwh'
})

In [98]:
df_unsorted['date'] = df_unsorted['timestamp'].dt.date
df_unsorted['time'] = df_unsorted['timestamp'].dt.time

In [99]:
df_unsorted = df_unsorted[['household_id', 'timestamp', 'date', 'time', 'consumption_kwh']]

In [100]:
df_unsorted_daylight_removed = df_unsorted.drop_duplicates(
    subset=['timestamp', 'household_id'],
    keep='last'
)

In [101]:
print(f"Total rows: {len(df_unsorted)}")
print(f"Total rows daylight: {len(df_unsorted_daylight_removed)}")

Total rows: 1264768
Total rows daylight: 1264608


In [102]:
df_unsorted_daylight_removed

,household_id,timestamp,date,time,consumption_kwh
743,s-1,2025-04-01 00:00:00,2025-04-01,00:00:00,0.026
744,s-1,2025-04-01 00:15:00,2025-04-01,00:15:00,0.063
745,s-1,2025-04-01 00:30:00,2025-04-01,00:30:00,0.021
746,s-1,2025-04-01 00:45:00,2025-04-01,00:45:00,0.028
747,s-1,2025-04-01 01:00:00,2025-04-01,01:00:00,0.013
...,...,...,...,...,...
1293740,xl-10,2026-02-28 22:45:00,2026-02-28,22:45:00,5.487
1293741,xl-10,2026-02-28 23:00:00,2026-02-28,23:00:00,0.798
1293742,xl-10,2026-02-28 23:15:00,2026-02-28,23:15:00,5.474
1293743,xl-10,2026-02-28 23:30:00,2026-02-28,23:30:00,5.366


In [103]:
df_unsorted_daylight_removed['date'] = df_unsorted_daylight_removed['timestamp'].dt.strftime('%d.%m.%Y')
df_unsorted_daylight_removed['timestamp'] = df_unsorted_daylight_removed['timestamp'].dt.strftime('%d.%m.%Y %H:%M:%S')

In [104]:
df_unsorted_daylight_removed

,household_id,timestamp,date,time,consumption_kwh
743,s-1,01.04.2025 00:00:00,01.04.2025,00:00:00,0.026
744,s-1,01.04.2025 00:15:00,01.04.2025,00:15:00,0.063
745,s-1,01.04.2025 00:30:00,01.04.2025,00:30:00,0.021
746,s-1,01.04.2025 00:45:00,01.04.2025,00:45:00,0.028
747,s-1,01.04.2025 01:00:00,01.04.2025,01:00:00,0.013
...,...,...,...,...,...
1293740,xl-10,28.02.2026 22:45:00,28.02.2026,22:45:00,5.487
1293741,xl-10,28.02.2026 23:00:00,28.02.2026,23:00:00,0.798
1293742,xl-10,28.02.2026 23:15:00,28.02.2026,23:15:00,5.474
1293743,xl-10,28.02.2026 23:30:00,28.02.2026,23:30:00,5.366


In [105]:
df_unsorted_daylight_removed.dtypes

household_id           str
timestamp              str
date                   str
time                object
consumption_kwh    float64
dtype: object

In [ ]:
#df_unsorted_daylight_removed.to_csv('sunly_consumption_sorted.csv', index=False)